In [1]:
# ============================================================
# ZapKart Dark Store Intelligence
# Notebook 5: Excel Operational Reports
# ============================================================

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (
    PatternFill, Font, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.chart.series import DataPoint
import psycopg2
from sqlalchemy import create_engine, text
import warnings
import os
warnings.filterwarnings('ignore')

PROCESSED_PATH = r"D:\Projects\End-to-end projects\9. ZapKart Dark-Store Intelligence\Data\Processed" + "\\"
EXCEL_PATH     = r"D:\Projects\End-to-end projects\9. ZapKart Dark-Store Intelligence\Excel" + "\\"
os.makedirs(EXCEL_PATH, exist_ok=True)

# Connect to PostgreSQL
engine = create_engine(
    "postgresql+psycopg2://",
    creator=lambda: psycopg2.connect(
        host     = "localhost",
        port     = 5432,
        dbname   = "zapkart_db",
        user     = "postgres",
        password = "NewStrongPassword@123"
    )
)

with engine.connect() as conn:
    print("✅ Connected to PostgreSQL")

print("✅ Excel output path set")
print("✅ All libraries loaded")

✅ Connected to PostgreSQL
✅ Excel output path set
✅ All libraries loaded


In [2]:
# ============================================================
# Pull all analysis results from PostgreSQL
# ============================================================

with engine.connect() as conn:

    # 1. Dark Store Scorecard
    scorecard = pd.read_sql("""
        SELECT
            s.store_name,
            s.city,
            m.total_orders,
            ROUND(CAST(m.total_revenue AS NUMERIC) / 100000, 1)
                                                          AS revenue_lac,
            ROUND(CAST(m.avg_delivery_mins AS NUMERIC), 2)
                                                          AS avg_delivery_mins,
            ROUND(CAST(m.sla_pct AS NUMERIC), 1)          AS sla_pct,
            ROUND(CAST(m.avg_order_value AS NUMERIC), 0)  AS avg_order_value,
            ROUND(CAST(m.avg_profit_per_order AS NUMERIC), 2)
                                                          AS avg_profit_per_order,
            RANK() OVER (ORDER BY m.total_revenue DESC)   AS revenue_rank,
            RANK() OVER (ORDER BY m.sla_pct DESC)         AS sla_rank,
            RANK() OVER (ORDER BY m.avg_profit_per_order DESC)
                                                          AS profit_rank,
            ROUND(CAST(
                RANK() OVER (ORDER BY m.total_revenue DESC) +
                RANK() OVER (ORDER BY m.sla_pct DESC) +
                RANK() OVER (ORDER BY m.avg_profit_per_order DESC)
            AS NUMERIC) / 3.0, 1)                         AS composite_score
        FROM (
            SELECT
                o.store_id,
                COUNT(o.order_id)                         AS total_orders,
                SUM(o.net_order_value_inr)                AS total_revenue,
                AVG(o.total_delivery_mins)                AS avg_delivery_mins,
                CAST(SUM(CASE WHEN o.sla_met = TRUE
                              THEN 1 ELSE 0 END) AS NUMERIC)
                    * 100.0 / NULLIF(COUNT(o.order_id),0) AS sla_pct,
                AVG(o.net_order_value_inr)                AS avg_order_value,
                AVG(o.gross_profit_inr)                   AS avg_profit_per_order
            FROM fact_orders o
            WHERE o.order_status = 'Delivered'
            GROUP BY o.store_id
        ) m
        JOIN dim_stores s ON m.store_id = s.store_id
        ORDER BY composite_score ASC
    """, conn)

    # 2. Store P&L Summary
    pnl = pd.read_sql("""
        SELECT
            s.store_name,
            s.city,
            ROUND(CAST(AVG(monthly_revenue) AS NUMERIC), 0)
                                                          AS avg_monthly_revenue,
            ROUND(CAST(AVG(monthly_profit) AS NUMERIC), 0)
                                                          AS avg_monthly_profit,
            s.monthly_fixed_cost_inr                      AS fixed_cost,
            CASE
                WHEN AVG(monthly_profit) > 0      THEN 'PROFITABLE'
                WHEN AVG(monthly_profit) > -50000 THEN 'MARGINAL'
                ELSE 'LOSS MAKING'
            END                                           AS store_status
        FROM (
            SELECT
                o.store_id,
                DATE_TRUNC('month', o.order_date::DATE)   AS month,
                SUM(o.net_order_value_inr)                AS monthly_revenue,
                CAST(SUM(o.gross_profit_inr) AS NUMERIC)
                - MAX(s2.monthly_fixed_cost_inr)          AS monthly_profit
            FROM fact_orders o
            JOIN dim_stores s2 ON o.store_id = s2.store_id
            WHERE o.order_status = 'Delivered'
            GROUP BY o.store_id,
                     DATE_TRUNC('month', o.order_date::DATE)
        ) monthly
        JOIN dim_stores s ON monthly.store_id = s.store_id
        GROUP BY s.store_name, s.city, s.monthly_fixed_cost_inr
        ORDER BY avg_monthly_profit DESC
    """, conn)

    # 3. SLA by Store
    sla = pd.read_sql("""
        SELECT
            s.store_name,
            s.city,
            COUNT(o.order_id)                             AS total_orders,
            ROUND(CAST(AVG(o.total_delivery_mins) AS NUMERIC), 2)
                                                          AS avg_delivery_mins,
            ROUND(CAST(AVG(o.pick_time_mins) AS NUMERIC), 2)
                                                          AS avg_pick_mins,
            ROUND(CAST(AVG(o.travel_time_mins) AS NUMERIC), 2)
                                                          AS avg_travel_mins,
            ROUND(
                CAST(SUM(CASE WHEN o.sla_met = TRUE
                              THEN 1 ELSE 0 END) AS NUMERIC)
                * 100.0 / NULLIF(COUNT(o.order_id), 0)
            , 2)                                          AS sla_pct
        FROM fact_orders o
        JOIN dim_stores s ON o.store_id = s.store_id
        WHERE o.order_status = 'Delivered'
        GROUP BY s.store_name, s.city
        ORDER BY sla_pct ASC
    """, conn)

    # 4. Picker Performance
    pickers = pd.read_sql("""
        SELECT
            p.picker_name,
            p.skill_level,
            p.shift,
            s.store_name,
            COUNT(pa.activity_id)                         AS total_picks,
            ROUND(CAST(AVG(pa.pick_rate_per_min) AS NUMERIC), 3)
                                                          AS avg_pick_rate,
            CAST(SUM(pa.errors_made) AS INT)              AS total_errors,
            ROUND(
                CAST(SUM(pa.errors_made) AS NUMERIC) * 100.0
                / NULLIF(CAST(COUNT(pa.activity_id) AS NUMERIC), 0)
            , 2)                                          AS error_rate_pct
        FROM fact_picker_activity pa
        JOIN dim_pickers p  ON pa.picker_id = p.picker_id
        JOIN dim_stores  s  ON pa.store_id  = s.store_id
        GROUP BY p.picker_name, p.skill_level,
                 p.shift, s.store_name
        ORDER BY avg_pick_rate DESC
    """, conn)

    # 5. Category Performance
    category = pd.read_sql("""
        SELECT
            oi.category,
            CAST(SUM(oi.quantity) AS INT)                 AS total_qty,
            ROUND(CAST(SUM(oi.item_total_inr) AS NUMERIC), 0)
                                                          AS total_revenue,
            ROUND(CAST(SUM(oi.item_margin_inr) AS NUMERIC), 0)
                                                          AS total_margin,
            ROUND(CAST(AVG(oi.item_margin_pct) AS NUMERIC) * 100, 1)
                                                          AS avg_margin_pct
        FROM fact_order_items oi
        GROUP BY oi.category
        ORDER BY total_revenue DESC
    """, conn)

    # 6. Hourly Demand
    hourly = pd.read_sql("""
        SELECT
            order_hour,
            COUNT(order_id)                               AS total_orders,
            ROUND(CAST(AVG(total_delivery_mins) AS NUMERIC), 2)
                                                          AS avg_delivery_mins,
            ROUND(
                CAST(SUM(CASE WHEN sla_met = TRUE
                              THEN 1 ELSE 0 END) AS NUMERIC)
                * 100.0 / NULLIF(COUNT(order_id), 0)
            , 2)                                          AS sla_pct
        FROM fact_orders
        WHERE order_status = 'Delivered'
        GROUP BY order_hour
        ORDER BY order_hour
    """, conn)

    # 7. Substitution Summary
    subs = pd.read_sql("""
        SELECT
            original_category,
            same_category,
            reason_for_sub,
            COUNT(substitution_id)                        AS total_subs,
            ROUND(
                CAST(SUM(CASE WHEN customer_accepted = TRUE
                              THEN 1 ELSE 0 END) AS NUMERIC)
                * 100.0 / NULLIF(COUNT(substitution_id), 0)
            , 2)                                          AS acceptance_rate_pct
        FROM fact_substitutions
        GROUP BY original_category, same_category, reason_for_sub
        ORDER BY acceptance_rate_pct DESC
    """, conn)

print("✅ All 7 datasets pulled from PostgreSQL")
print(f"   Scorecard:  {len(scorecard)} stores")
print(f"   P&L:        {len(pnl)} stores")
print(f"   SLA:        {len(sla)} stores")
print(f"   Pickers:    {len(pickers)} pickers")
print(f"   Categories: {len(category)} categories")
print(f"   Hourly:     {len(hourly)} hours")
print(f"   Subs:       {len(subs)} rows")

✅ All 7 datasets pulled from PostgreSQL
   Scorecard:  10 stores
   P&L:        10 stores
   SLA:        10 stores
   Pickers:    80 pickers
   Categories: 8 categories
   Hourly:     24 hours
   Subs:       48 rows


In [3]:
# ============================================================
# Define reusable styles for professional formatting
# ============================================================

# Colors
DARK_BLUE   = "1E3A5F"
MID_BLUE    = "2563EB"
LIGHT_BLUE  = "DBEAFE"
WHITE       = "FFFFFF"
DARK_GRAY   = "374151"
LIGHT_GRAY  = "F9FAFB"
GREEN       = "16A34A"
LIGHT_GREEN = "DCFCE7"
RED         = "DC2626"
LIGHT_RED   = "FEE2E2"
AMBER       = "D97706"
LIGHT_AMBER = "FEF3C7"
YELLOW      = "FFF9C4"

def header_fill(color=DARK_BLUE):
    return PatternFill("solid", fgColor=color)

def cell_fill(color=LIGHT_BLUE):
    return PatternFill("solid", fgColor=color)

def header_font(size=11, bold=True, color=WHITE):
    return Font(name="Calibri", size=size,
                bold=bold, color=color)

def body_font(size=10, bold=False, color=DARK_GRAY):
    return Font(name="Calibri", size=size,
                bold=bold, color=color)

def center_align():
    return Alignment(horizontal="center",
                     vertical="center", wrap_text=True)

def left_align():
    return Alignment(horizontal="left",
                     vertical="center", wrap_text=True)

def thin_border():
    thin = Side(style="thin", color="D1D5DB")
    return Border(left=thin, right=thin,
                  top=thin, bottom=thin)

def write_df_to_sheet(ws, df, start_row=1,
                       header_color=DARK_BLUE):
    """Write a DataFrame to a worksheet with full formatting."""
    cols = df.columns.tolist()

    # Write headers
    for col_idx, col_name in enumerate(cols, 1):
        cell = ws.cell(row=start_row,
                        column=col_idx,
                        value=str(col_name).upper()
                              .replace("_", " "))
        cell.fill      = header_fill(header_color)
        cell.font      = header_font()
        cell.alignment = center_align()
        cell.border    = thin_border()

    # Write data rows
    for row_idx, row in enumerate(
            df.itertuples(index=False), start_row + 1):
        for col_idx, value in enumerate(row, 1):
            cell = ws.cell(row=row_idx,
                            column=col_idx,
                            value=value)
            cell.font      = body_font()
            cell.alignment = center_align()
            cell.border    = thin_border()
            # Alternate row shading
            if (row_idx - start_row) % 2 == 0:
                cell.fill = cell_fill(LIGHT_GRAY)

    # Auto-fit column widths
    for col_idx, col_name in enumerate(cols, 1):
        col_letter = get_column_letter(col_idx)
        max_len    = max(
            len(str(col_name)),
            df[col_name].astype(str).str.len().max()
        ) + 4
        ws.column_dimensions[col_letter].width = min(max_len, 30)

    return ws

print("✅ Excel styles defined")

✅ Excel styles defined


In [4]:
# ============================================================
# Build the complete Excel workbook
# ============================================================

wb = Workbook()
wb.remove(wb.active)  # Remove default blank sheet

print("Building Excel workbook...")

Building Excel workbook...


In [5]:
# ============================================================
# SHEET 1: Executive Summary
# ============================================================

ws1 = wb.create_sheet("Executive Summary")
ws1.sheet_view.showGridLines = False
ws1.column_dimensions['A'].width = 35
ws1.column_dimensions['B'].width = 25
ws1.column_dimensions['C'].width = 25
ws1.column_dimensions['D'].width = 25

# Title block
ws1.merge_cells("A1:D1")
title_cell        = ws1["A1"]
title_cell.value  = "ZAPKART DARK STORE INTELLIGENCE"
title_cell.fill   = header_fill(DARK_BLUE)
title_cell.font   = Font(name="Calibri", size=18,
                          bold=True, color=WHITE)
title_cell.alignment = center_align()
ws1.row_dimensions[1].height = 45

ws1.merge_cells("A2:D2")
sub_cell        = ws1["A2"]
sub_cell.value  = "Executive Summary Report — Jan 2023 to Jun 2024"
sub_cell.fill   = header_fill(MID_BLUE)
sub_cell.font   = Font(name="Calibri", size=12,
                        bold=False, color=WHITE)
sub_cell.alignment = center_align()
ws1.row_dimensions[2].height = 25

# KPI Section Header
ws1.merge_cells("A4:D4")
kpi_header        = ws1["A4"]
kpi_header.value  = "KEY PERFORMANCE INDICATORS"
kpi_header.fill   = header_fill(DARK_GRAY)
kpi_header.font   = header_font(size=12)
kpi_header.alignment = center_align()
ws1.row_dimensions[4].height = 30

# Calculate KPIs from dataframes
total_revenue    = float(scorecard['revenue_lac'].sum())
avg_sla          = float(scorecard['sla_pct'].mean())
avg_delivery     = float(scorecard['avg_delivery_mins'].mean())
total_orders     = int(scorecard['total_orders'].sum())
profitable_stores = len(pnl[pnl['store_status'] == 'PROFITABLE'])
avg_order_val    = float(scorecard['avg_order_value'].mean())

kpis = [
    ("Total Revenue (18 Months)",
     f"Rs {total_revenue:.1f} Lakhs",   LIGHT_BLUE,  MID_BLUE),
    ("Total Orders Delivered",
     f"{total_orders:,}",               LIGHT_GREEN, GREEN),
    ("Network SLA Achievement",
     f"{avg_sla:.1f}%",
     LIGHT_GREEN if avg_sla >= 85
     else LIGHT_RED,
     GREEN if avg_sla >= 85 else RED),
    ("Avg Delivery Time",
     f"{avg_delivery:.1f} mins",
     LIGHT_GREEN if avg_delivery <= 10
     else LIGHT_AMBER,
     GREEN if avg_delivery <= 10 else AMBER),
    ("Avg Order Value",
     f"Rs {avg_order_val:,.0f}",        LIGHT_BLUE,  MID_BLUE),
    ("Profitable Stores",
     f"{profitable_stores} of 10",
     LIGHT_GREEN if profitable_stores >= 7
     else LIGHT_RED,
     GREEN if profitable_stores >= 7 else RED),
]

for i, (label, value, bg, fg) in enumerate(kpis, 5):
    ws1.row_dimensions[i].height = 32
    ws1.merge_cells(f"A{i}:B{i}")
    label_cell        = ws1[f"A{i}"]
    label_cell.value  = label
    label_cell.fill   = cell_fill(bg)
    label_cell.font   = Font(name="Calibri", size=11,
                              bold=True, color=DARK_GRAY)
    label_cell.alignment = left_align()
    label_cell.border = thin_border()

    ws1.merge_cells(f"C{i}:D{i}")
    value_cell        = ws1[f"C{i}"]
    value_cell.value  = value
    value_cell.fill   = cell_fill(bg)
    value_cell.font   = Font(name="Calibri", size=13,
                              bold=True, color=fg)
    value_cell.alignment = center_align()
    value_cell.border = thin_border()

# Store Status Summary
row = len(kpis) + 7
ws1.merge_cells(f"A{row}:D{row}")
status_header        = ws1[f"A{row}"]
status_header.value  = "STORE STATUS SUMMARY"
status_header.fill   = header_fill(DARK_GRAY)
status_header.font   = header_font(size=12)
status_header.alignment = center_align()
ws1.row_dimensions[row].height = 30

status_counts = pnl['store_status'].value_counts()
status_colors = {
    'PROFITABLE':  (LIGHT_GREEN, GREEN),
    'MARGINAL':    (LIGHT_AMBER, AMBER),
    'LOSS MAKING': (LIGHT_RED,   RED)
}

for j, (status, count) in enumerate(
        status_counts.items(), row + 1):
    bg, fg = status_colors.get(status, (LIGHT_GRAY, DARK_GRAY))
    ws1.row_dimensions[j].height = 28
    ws1.merge_cells(f"A{j}:B{j}")
    s_label        = ws1[f"A{j}"]
    s_label.value  = status
    s_label.fill   = cell_fill(bg)
    s_label.font   = Font(name="Calibri", size=11,
                           bold=True, color=fg)
    s_label.alignment = center_align()
    s_label.border = thin_border()

    ws1.merge_cells(f"C{j}:D{j}")
    s_value        = ws1[f"C{j}"]
    s_value.value  = f"{count} Stores"
    s_value.fill   = cell_fill(bg)
    s_value.font   = Font(name="Calibri", size=12,
                           bold=True, color=fg)
    s_value.alignment = center_align()
    s_value.border = thin_border()

print("✅ Sheet 1: Executive Summary built")

✅ Sheet 1: Executive Summary built


In [6]:
# ============================================================
# SHEET 2: Dark Store Scorecard
# ============================================================

ws2 = wb.create_sheet("Dark Store Scorecard")
ws2.sheet_view.showGridLines = False

# Title
ws2.merge_cells("A1:L1")
t = ws2["A1"]
t.value     = "DARK STORE SCORECARD — All 10 Stores Ranked"
t.fill      = header_fill(DARK_BLUE)
t.font      = Font(name="Calibri", size=14,
                    bold=True, color=WHITE)
t.alignment = center_align()
ws2.row_dimensions[1].height = 38

write_df_to_sheet(ws2, scorecard, start_row=3)

# Color code composite score column (last column)
score_col = len(scorecard.columns)
for row_idx in range(4, 4 + len(scorecard)):
    cell = ws2.cell(row=row_idx, column=score_col)
    val  = cell.value
    if val is not None:
        if float(val) <= 3.0:
            cell.fill = cell_fill(LIGHT_GREEN)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=GREEN)
        elif float(val) <= 6.0:
            cell.fill = cell_fill(LIGHT_AMBER)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=AMBER)
        else:
            cell.fill = cell_fill(LIGHT_RED)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=RED)

# Color code SLA column
sla_col = scorecard.columns.tolist().index('sla_pct') + 1
for row_idx in range(4, 4 + len(scorecard)):
    cell = ws2.cell(row=row_idx, column=sla_col)
    val  = cell.value
    if val is not None:
        if float(val) >= 88:
            cell.fill = cell_fill(LIGHT_GREEN)
        elif float(val) >= 80:
            cell.fill = cell_fill(LIGHT_AMBER)
        else:
            cell.fill = cell_fill(LIGHT_RED)

print("✅ Sheet 2: Dark Store Scorecard built")

✅ Sheet 2: Dark Store Scorecard built


In [7]:
# ============================================================
# SHEET 3: Store P&L Report
# ============================================================

ws3 = wb.create_sheet("Store P&L")
ws3.sheet_view.showGridLines = False

ws3.merge_cells("A1:F1")
t = ws3["A1"]
t.value     = "STORE PROFIT & LOSS REPORT — Monthly Average"
t.fill      = header_fill(DARK_BLUE)
t.font      = Font(name="Calibri", size=14,
                    bold=True, color=WHITE)
t.alignment = center_align()
ws3.row_dimensions[1].height = 38

write_df_to_sheet(ws3, pnl, start_row=3)

# Color code store_status and avg_monthly_profit columns
status_col = pnl.columns.tolist().index('store_status') + 1
profit_col = pnl.columns.tolist().index('avg_monthly_profit') + 1

for row_idx in range(4, 4 + len(pnl)):
    status_cell = ws3.cell(row=row_idx, column=status_col)
    profit_cell = ws3.cell(row=row_idx, column=profit_col)

    status = status_cell.value
    if status == 'PROFITABLE':
        status_cell.fill = cell_fill(LIGHT_GREEN)
        status_cell.font = Font(name="Calibri", size=10,
                                 bold=True, color=GREEN)
        profit_cell.fill = cell_fill(LIGHT_GREEN)
    elif status == 'MARGINAL':
        status_cell.fill = cell_fill(LIGHT_AMBER)
        status_cell.font = Font(name="Calibri", size=10,
                                 bold=True, color=AMBER)
        profit_cell.fill = cell_fill(LIGHT_AMBER)
    elif status == 'LOSS MAKING':
        status_cell.fill = cell_fill(LIGHT_RED)
        status_cell.font = Font(name="Calibri", size=10,
                                 bold=True, color=RED)
        profit_cell.fill = cell_fill(LIGHT_RED)

print("✅ Sheet 3: Store P&L built")

✅ Sheet 3: Store P&L built


In [8]:
# ============================================================
# SHEET 4: SLA Performance Report
# ============================================================

ws4 = wb.create_sheet("SLA Performance")
ws4.sheet_view.showGridLines = False

ws4.merge_cells("A1:G1")
t = ws4["A1"]
t.value     = "SLA PERFORMANCE REPORT — 10-Minute Delivery Promise"
t.fill      = header_fill(DARK_BLUE)
t.font      = Font(name="Calibri", size=14,
                    bold=True, color=WHITE)
t.alignment = center_align()
ws4.row_dimensions[1].height = 38

# Add SLA benchmark note
ws4.merge_cells("A2:G2")
note        = ws4["A2"]
note.value  = ("Target: >= 88% SLA  |  "
               "Warning: 80-88%  |  "
               "Critical: < 80%")
note.fill   = cell_fill(LIGHT_AMBER)
note.font   = Font(name="Calibri", size=10,
                    bold=True, color=AMBER)
note.alignment = center_align()
ws4.row_dimensions[2].height = 22

write_df_to_sheet(ws4, sla, start_row=4)

# Color code SLA % column
sla_col_idx = sla.columns.tolist().index('sla_pct') + 1
for row_idx in range(5, 5 + len(sla)):
    cell = ws4.cell(row=row_idx, column=sla_col_idx)
    val  = cell.value
    if val is not None:
        fval = float(val)
        if fval >= 88:
            cell.fill = cell_fill(LIGHT_GREEN)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=GREEN)
        elif fval >= 80:
            cell.fill = cell_fill(LIGHT_AMBER)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=AMBER)
        else:
            cell.fill = cell_fill(LIGHT_RED)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=RED)

print("✅ Sheet 4: SLA Performance built")

✅ Sheet 4: SLA Performance built


In [9]:
# ============================================================
# SHEET 5: Picker Performance
# ============================================================

ws5 = wb.create_sheet("Picker Performance")
ws5.sheet_view.showGridLines = False

ws5.merge_cells("A1:H1")
t = ws5["A1"]
t.value     = "PICKER PRODUCTIVITY REPORT — All 80 Pickers"
t.fill      = header_fill(DARK_BLUE)
t.font      = Font(name="Calibri", size=14,
                    bold=True, color=WHITE)
t.alignment = center_align()
ws5.row_dimensions[1].height = 38

write_df_to_sheet(ws5, pickers, start_row=3)

# Color code pick rate column
rate_col = pickers.columns.tolist().index('avg_pick_rate') + 1
network_avg = float(pickers['avg_pick_rate'].mean())

for row_idx in range(4, 4 + len(pickers)):
    cell = ws5.cell(row=row_idx, column=rate_col)
    val  = cell.value
    if val is not None:
        fval = float(val)
        if fval >= network_avg * 1.1:
            cell.fill = cell_fill(LIGHT_GREEN)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=GREEN)
        elif fval < network_avg * 0.9:
            cell.fill = cell_fill(LIGHT_RED)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=RED)

# Color code error rate column
err_col = pickers.columns.tolist().index('error_rate_pct') + 1
for row_idx in range(4, 4 + len(pickers)):
    cell = ws5.cell(row=row_idx, column=err_col)
    val  = cell.value
    if val is not None:
        if float(val) > 10:
            cell.fill = cell_fill(LIGHT_RED)

print("✅ Sheet 5: Picker Performance built")

# ============================================================
# SHEET 6: Category Performance
# ============================================================

ws6 = wb.create_sheet("Category Performance")
ws6.sheet_view.showGridLines = False

ws6.merge_cells("A1:E1")
t = ws6["A1"]
t.value     = "CATEGORY PERFORMANCE REPORT"
t.fill      = header_fill(DARK_BLUE)
t.font      = Font(name="Calibri", size=14,
                    bold=True, color=WHITE)
t.alignment = center_align()
ws6.row_dimensions[1].height = 38

write_df_to_sheet(ws6, category, start_row=3)

# Color code margin column
margin_col = category.columns.tolist().index('avg_margin_pct') + 1
for row_idx in range(4, 4 + len(category)):
    cell = ws6.cell(row=row_idx, column=margin_col)
    val  = cell.value
    if val is not None:
        if float(val) >= 25:
            cell.fill = cell_fill(LIGHT_GREEN)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=GREEN)
        elif float(val) < 18:
            cell.fill = cell_fill(LIGHT_RED)

print("✅ Sheet 6: Category Performance built")

# ============================================================
# SHEET 7: Hourly Demand Pattern
# ============================================================

ws7 = wb.create_sheet("Hourly Demand")
ws7.sheet_view.showGridLines = False

ws7.merge_cells("A1:D1")
t = ws7["A1"]
t.value     = "HOURLY DEMAND PATTERN — Picker Shift Planning"
t.fill      = header_fill(DARK_BLUE)
t.font      = Font(name="Calibri", size=14,
                    bold=True, color=WHITE)
t.alignment = center_align()
ws7.row_dimensions[1].height = 38

write_df_to_sheet(ws7, hourly, start_row=3)

# Highlight peak hours
order_col = hourly.columns.tolist().index('total_orders') + 1
p75       = hourly['total_orders'].quantile(0.75)
p90       = hourly['total_orders'].quantile(0.90)

for row_idx in range(4, 4 + len(hourly)):
    cell = ws7.cell(row=row_idx, column=order_col)
    val  = cell.value
    if val is not None:
        ival = int(val)
        if ival >= p90:
            cell.fill = cell_fill(LIGHT_RED)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=RED)
        elif ival >= p75:
            cell.fill = cell_fill(LIGHT_AMBER)

print("✅ Sheet 7: Hourly Demand built")

# ============================================================
# SHEET 8: Substitution Analysis
# ============================================================

ws8 = wb.create_sheet("Substitution Analysis")
ws8.sheet_view.showGridLines = False

ws8.merge_cells("A1:E1")
t = ws8["A1"]
t.value     = "SUBSTITUTION ACCEPTANCE ANALYSIS"
t.fill      = header_fill(DARK_BLUE)
t.font      = Font(name="Calibri", size=14,
                    bold=True, color=WHITE)
t.alignment = center_align()
ws8.row_dimensions[1].height = 38

write_df_to_sheet(ws8, subs, start_row=3)

# Color code acceptance rate
acc_col = subs.columns.tolist().index('acceptance_rate_pct') + 1
for row_idx in range(4, 4 + len(subs)):
    cell = ws8.cell(row=row_idx, column=acc_col)
    val  = cell.value
    if val is not None:
        fval = float(val)
        if fval >= 70:
            cell.fill = cell_fill(LIGHT_GREEN)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=GREEN)
        elif fval < 50:
            cell.fill = cell_fill(LIGHT_RED)
            cell.font = Font(name="Calibri", size=10,
                              bold=True, color=RED)
        else:
            cell.fill = cell_fill(LIGHT_AMBER)

print("✅ Sheet 8: Substitution Analysis built")

✅ Sheet 5: Picker Performance built
✅ Sheet 6: Category Performance built
✅ Sheet 7: Hourly Demand built
✅ Sheet 8: Substitution Analysis built


In [10]:
# ============================================================
# Save the complete workbook
# ============================================================

output_file = f"{EXCEL_PATH}ZapKart_Operational_Reports.xlsx"
wb.save(output_file)

print("=" * 60)
print("ZAPKART EXCEL REPORT — COMPLETE")
print("=" * 60)
print(f"  File saved: ZapKart_Operational_Reports.xlsx")
print(f"  Location:   {EXCEL_PATH}")
print(f"  Sheets:     8 sheets")
print("=" * 60)
print("\n  Sheet 1 — Executive Summary")
print("  Sheet 2 — Dark Store Scorecard")
print("  Sheet 3 — Store P&L")
print("  Sheet 4 — SLA Performance")
print("  Sheet 5 — Picker Performance")
print("  Sheet 6 — Category Performance")
print("  Sheet 7 — Hourly Demand")
print("  Sheet 8 — Substitution Analysis")
print("=" * 60)
print("\n🎉 Excel Report Complete. Ready for Step 7: Power BI.")

ZAPKART EXCEL REPORT — COMPLETE
  File saved: ZapKart_Operational_Reports.xlsx
  Location:   D:\Projects\End-to-end projects\9. ZapKart Dark-Store Intelligence\Excel\
  Sheets:     8 sheets

  Sheet 1 — Executive Summary
  Sheet 2 — Dark Store Scorecard
  Sheet 3 — Store P&L
  Sheet 4 — SLA Performance
  Sheet 5 — Picker Performance
  Sheet 6 — Category Performance
  Sheet 7 — Hourly Demand
  Sheet 8 — Substitution Analysis

🎉 Excel Report Complete. Ready for Step 7: Power BI.
